# Create unified output file

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../../data'

OUTPUT_PATH = os.path.join(DATA_PATH, 'lme_ready')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_PATH, 'resids.parquet')
print(bool(os.path.exists(FINAL_DOC)))

False


In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('._'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
files = [f for f in grab_all_files(OUTPUT_PATH, '.tsv') if (not f.startswith('._'))]
print(files)

['../data/lme_ready/cha-croatian-ceda-analysis.tsv', '../data/lme_ready/cha-yiddish-ceda-analysis.tsv', '../data/lme_ready/cha-callfriend-ceda-analysis.tsv', '../data/lme_ready/cha-callhome-ceda-analysis.tsv', '../data/lme_ready/cha-finchat_corpus-ceda-analysis.tsv', '../data/lme_ready/cha-ko_corpus-ceda-analysis.tsv', '../data/lme_ready/cha-null-ceda-analysis.tsv']


## Processing the data

Everything so far is saved into individual documents. This takes all of that and merges them into a single over-arching document, which is slightly less unwieldy.

In [5]:
start_from_turn = 5
min_acceptable_tokens = 5

In [6]:
df = []

In [7]:
for f in tqdm(files):
    print('\n', f)
    df += [pd.read_table(f, sep='\t')]

    if '/cha-null-' in f:
        df[-1]['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df[-1]['x_speaker'])]
        df[-1]['x_turn_id'] = [int(i.split('-')[-1]) for i in df[-1]['x_turn_id']]

    elif ('/cha-' in f) or ('grace-' in f):
        df[-1]['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df[-1]['x_speaker'])]

    elif '/candor-' in f:
        df[-1]['conversation_id'] = [sp.replace('.csv', '') for sp in tqdm(df[-1]['conversation_id'])]
        df[-1]['corpus'] = 'CANDOR'

    else:
        if 'conversation_id' in df[-1].columns:
            0
        else:
            df[-1]['conversation_id'] = f

    df[-1] = df[-1].loc[
        (df[-1]['nx'] >= min_acceptable_tokens)
        & (df[-1]['ny'] >= min_acceptable_tokens)
        & (df[-1]['x_turn_id'] >= start_from_turn)
    ]

    df[-1]['dyad'] = (df[-1]['x_speaker'] + '-' + df[-1]['y_speaker']).replace('\t', '-')
    df[-1]['turn_delta'] = (df[-1]['y_turn_id'] - df[-1]['x_turn_id'])
    df[-1] = df[-1][['Hxy', 'nx', 'ny', 'turn_delta', 'self',
                     'x_speaker',
                     'dyad', 'corpus', 'conversation_id', 'x_turn_id']]

    ##################################################################
    ### turning x_turn_id into a unique indicator per each utterance
    ##################################################################
    df[-1]['x_turn_id'] = df[-1]['conversation_id'].astype(str) + '-' + df[-1]['x_turn_id'].astype(str)

    df[-1]['null'] = bool('null-' in f)
df = pd.concat(df, ignore_index=True)

df.shape

  0%|          | 0/7 [00:00<?, ?it/s]


 ../data/lme_ready/cha-croatian-ceda-analysis.tsv


  0%|          | 0/3311067 [00:00<?, ?it/s]


 ../data/lme_ready/cha-yiddish-ceda-analysis.tsv


  0%|          | 0/13121 [00:00<?, ?it/s]

...


 ../data/lme_ready/cha-ko_corpus-ceda-analysis.tsv


  0%|          | 0/2197289 [00:00<?, ?it/s]


 ../data/lme_ready/cha-null-ceda-analysis.tsv


  0%|          | 0/2560745 [00:00<?, ?it/s]

(17499899, 11)

In [8]:
df['null'].value_counts()

null
False    16056138
True      1443761
Name: count, dtype: int64

Adding back in a langauge column indicator

In [9]:
# to rename the corpus correctly . . .
def lang(x):
    return x.split('-')[1]

In [10]:
df['lang'] = [lang(x) for x in tqdm(df['corpus'])]

  0%|          | 0/17499899 [00:00<?, ?it/s]

I had some worries about using the japanese dataset with `xlm-roberta-base`. [Some other users have noted that xlm-roberta pathologically renders CoS (thus CoE values) superficially high (low)](https://huggingface.co/FacebookAI/xlm-roberta-base/discussions/18) when comparing across sentences (in other words, the model exhibits high anisotropy for Japanese in particular).

It may be worth it to actually take this experiment and re-run all the vectors using a multilingual MPNET model. MPNET is architecturally similar to RoBERTa, but the cloze task was/is dependency aware, however. That may yield stronger interlanguage differences just by dint of it being a different model.

In [11]:
df = df.loc[~df['lang'].isin([
    'jpn',
    # 'fin', 'yid'
])]
df.shape

(15310889, 12)

In [12]:
df['conversation_id'].nunique()

1285

#### Checking on and dealing with null-distribution test results

In [13]:
df['null'].value_counts()

null
False    14065648
True      1245241
Name: count, dtype: int64

In [14]:
df['x_turn_id'].loc[df['null']] = 'null-'+ df['x_turn_id'].loc[df['null']]

#### other checks

In [15]:
df.isna().sum()

Hxy                0
nx                 0
ny                 0
turn_delta         0
self               0
x_speaker          0
dyad               0
corpus             0
conversation_id    0
x_turn_id          0
null               0
lang               0
dtype: int64

In [16]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,lang
0,2.765743,12.0,10.0,2.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro
1,2.836436,12.0,7.0,6.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro
2,2.906804,12.0,8.0,9.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro
3,3.063219,12.0,5.0,10.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro
4,2.971834,12.0,9.0,12.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro


## Relabel/add a column for per comparison sample $\Delta$

In my initial work, it didn't really matter what number sample each value was. So it didn't matter if $t \pm 1$ was a self or other comment. Who cared. But for merging the data and for calculating allan variance, this is actually kinda important. So now I'm re-adding this in there.

This all might be moot. There is very much a world in which what I actually need is a much simpler comparison between turn $\Delta$ values. Perhaps using Kolomogorov-Smirnov testing between each sample is sufficient? All I need, after all, is to prove that power-scaling is occurring here. And Allan-variance is just one test (though a well-trodden one) that can be used to assess that.

In [17]:
df['groups'] = df['x_turn_id'].astype(str)

In [18]:
df['sample_delta'] = df.groupby(by=['groups', 'self']).cumcount() + 1

In [19]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,lang,groups,sample_delta
0,2.765743,12.0,10.0,2.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,1
1,2.836436,12.0,7.0,6.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,2
2,2.906804,12.0,8.0,9.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,3
3,3.063219,12.0,5.0,10.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,4
4,2.971834,12.0,9.0,12.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,5


## Tidying the data

Making sure there aren't any erroneous `\t` elements in there.

In [20]:
# cleaning
for col in list(df):
    print(col)
    df[col] = [it.replace('\t', '') if isinstance(it,str) else it for it in tqdm(df[col])]

Hxy


  0%|          | 0/15310889 [00:00<?, ?it/s]

nx


  0%|          | 0/15310889 [00:00<?, ?it/s]

ny


...

  0%|          | 0/15310889 [00:00<?, ?it/s]

groups


  0%|          | 0/15310889 [00:00<?, ?it/s]

sample_delta


  0%|          | 0/15310889 [00:00<?, ?it/s]

## Save outputs

Because that's the whole f'ing point.

In [21]:
df.to_parquet(FINAL_DOC)

## Check if parquet works

In [22]:
df_ = pd.read_parquet(FINAL_DOC)

In [23]:
del df_